# CNN-EXP-007 — BPE-100 + pretokenized contigs

Новая версия эксперимента делает две вещи одновременно:

1. **BPE-50 → BPE-100** — проверяем более сильную компрессию/более длинные токены.
2. **Убираем BPE из training hot path** — каждый contig токенизируется один раз в PLUS и RC, затем training/validation используют только быстрые `slice + gather`.

Модель остаётся той же по идее: `LOCAL 1284 bp` + `GLOBAL 9216 bp → BPE → dilated CNN → fusion → presence/intensity`.

Для архитектурного sweep по умолчанию: **1000 steps** и фиксированный validation-subset. Полную validation запускаем только для финалистов.

> Full-contig BPE фиксирует сегментацию на всём contig. Поэтому токены непосредственно у границ 9216-bp окна могут отличаться от старой per-window BPE-токенизации. Центральные 1284 bp находятся примерно в 4 kb от границ, поэтому это приемлемая и осознанная часть нового pipeline.


In [ ]:
from pathlib import Path
import gc
import hashlib
import math
import re
import sys
import time
import warnings

from IPython.display import display
import pandas as pd
import torch
import torch.nn as nn
import torch_directml

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
    dml_weighted_huber,
    dml_weighted_profile_bce_with_logits,
)
from ml_project.epic_data import load_split

print("Project:", PROJECT_ROOT)


In [ ]:
device = torch_directml.device()
print("DirectML device:", device)
print("GPU:", torch_directml.device_name(0))


In [ ]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

train_contigs = tuple(sorted(set(split.intervals("train")["contig"].astype(str))))
validation_contigs = tuple(sorted(set(split.intervals("validation")["contig"].astype(str))))
assert set(train_contigs).isdisjoint(validation_contigs)

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

GLOBAL_FLANK = 4096
TARGET_LENGTH = profile_config.target_length
LOCAL_FLANK = profile_config.context_flank
GLOBAL_LENGTH = GLOBAL_FLANK + TARGET_LENGTH + GLOBAL_FLANK
LOCAL_LENGTH = LOCAL_FLANK + TARGET_LENGTH + LOCAL_FLANK
LOCAL_START_IN_GLOBAL = GLOBAL_FLANK - LOCAL_FLANK
LOCAL_END_IN_GLOBAL = LOCAL_START_IN_GLOBAL + LOCAL_LENGTH

train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)
validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

display(split.summary)
print("Train contigs:", len(train_contigs))
print("Validation contigs:", len(validation_contigs))
print("Local / Global:", LOCAL_LENGTH, "/", GLOBAL_LENGTH)


In [ ]:
# BPE-100 configuration.
# Если artifact уже существует — просто загрузим его.
# Если нет — следующая ячейка расширит существующий BPE-50 до 100 merges.
BPE_MERGES = 100
BPE50_PATH = PROJECT_ROOT / "artifacts" / "bpe_softmask_50.pt"
BPE_PATH = PROJECT_ROOT / "artifacts" / "bpe_softmask_100.pt"
BUILD_BPE_100_IF_MISSING = True

print("BPE-50:", BPE50_PATH)
print("BPE-100:", BPE_PATH)
print("BPE-100 exists:", BPE_PATH.is_file())


In [ ]:
# ============================================================
# OPTIONAL: EXTEND BPE-50 -> BPE-100
# ============================================================
# Это OFFLINE-шаг: выполняется один раз. Он torch-only, без NumPy BPE.
# Не запускай его параллельно со старой full validation, если не хочешь
# делить CPU/RAM между двумя notebook.


def _norm_rules(history):
    return [tuple(int(v) for v in rule) for rule in history]


def _single_merge(tokens, left, right, new_token):
    if tokens.numel() < 2:
        return tokens
    matches = (tokens[:-1] == left) & (tokens[1:] == right)
    starts = torch.nonzero(matches, as_tuple=False).flatten()
    if starts.numel() == 0:
        return tokens

    if left == right and starts.numel() > 1:
        new_run = torch.cat([
            torch.tensor([True], dtype=torch.bool),
            starts[1:] != starts[:-1] + 1,
        ])
        run_ids = torch.cumsum(new_run.long(), 0) - 1
        run_starts = torch.nonzero(new_run, as_tuple=False).flatten()
        offsets = torch.arange(starts.numel()) - run_starts[run_ids]
        starts = starts[offsets % 2 == 0]

    out = tokens.clone()
    out[starts] = new_token
    remove = torch.zeros(tokens.numel(), dtype=torch.bool)
    remove[starts + 1] = True
    return out[~remove]


def _apply_rules(tokens, rules):
    for left, right, new_token in rules:
        tokens = _single_merge(tokens, left, right, new_token)
    return tokens


def build_bpe100_from_bpe50():
    if not BPE50_PATH.is_file():
        raise FileNotFoundError(BPE50_PATH)

    base = torch.load(BPE50_PATH, map_location="cpu", weights_only=False)
    rules = _norm_rules(base["merge_history"])
    if len(rules) != 50:
        raise ValueError(f"Expected BPE-50, got {len(rules)} merges")

    token_text = {int(k): str(v) for k,v in base["token_text"].items()}
    token_length = {int(k): int(v) for k,v in base["token_length"].items()}

    # Replay first 50 merges once and keep the compact token streams in RAM.
    streams = {}
    total_bp = 0
    started = time.perf_counter()

    for i, contig in enumerate(train_contigs, 1):
        raw = torch.as_tensor(sequences[contig], dtype=torch.uint8)
        x = raw.to(torch.int16)
        tokens = torch.where(
            x < 8,
            torch.remainder(x, 4),
            torch.full_like(x, 4),
        ).to(torch.uint8)
        tokens = _apply_rules(tokens, rules).to(torch.uint8).contiguous()
        streams[contig] = tokens
        total_bp += raw.numel()
        print(f"BPE-50 replay {i:>3}/{len(train_contigs)} {contig}: {tokens.numel():,} tokens")

    PAIR_BASE = 256
    PAIR_SPACE = PAIR_BASE * PAIR_BASE

    while len(rules) < BPE_MERGES:
        counts = torch.zeros(PAIR_SPACE, dtype=torch.int64)
        for tokens in streams.values():
            if tokens.numel() < 2:
                continue
            pair_code = tokens[:-1].long() * PAIR_BASE + tokens[1:].long()
            counts += torch.bincount(pair_code, minlength=PAIR_SPACE)

        best = int(torch.argmax(counts))
        best_count = int(counts[best])
        left, right = divmod(best, PAIR_BASE)
        new_token = 5 + len(rules)
        if best_count <= 0:
            raise RuntimeError("No pair left to merge")
        if new_token >= 256:
            raise RuntimeError("uint8 token storage is no longer sufficient")

        for contig in streams:
            streams[contig] = _single_merge(
                streams[contig], left, right, new_token
            ).to(torch.uint8).contiguous()

        rules.append((left, right, new_token))
        token_text[new_token] = token_text[left] + token_text[right]
        token_length[new_token] = token_length[left] + token_length[right]

        total_tokens = sum(t.numel() for t in streams.values())
        print(
            f"merge={len(rules):3d} pair=({left},{right}) -> {new_token} "
            f"count={best_count:,} compression={total_bp/total_tokens:.4f}x"
        )

    artifact = {
        "merge_history": rules,
        "token_text": token_text,
        "token_length": token_length,
        "vocab_size": 5 + len(rules),
        "num_merges": len(rules),
        "train_bp": int(total_bp),
        "source_artifact": str(BPE50_PATH),
        "case_independent": True,
        "softmask_mode": "per-occurrence lowercase_count",
    }
    torch.save(artifact, BPE_PATH)
    print("Saved:", BPE_PATH)
    print("Minutes:", round((time.perf_counter()-started)/60, 2))


if not BPE_PATH.is_file():
    if BUILD_BPE_100_IF_MISSING:
        build_bpe100_from_bpe50()
    else:
        raise FileNotFoundError(BPE_PATH)
else:
    print("Using existing BPE-100 artifact")


In [ ]:
# Load BPE-100 and build small CPU lookup tables.
bpe_artifact = torch.load(BPE_PATH, map_location="cpu", weights_only=False)
merge_history = [tuple(int(v) for v in rule) for rule in bpe_artifact["merge_history"]]
token_length = {int(k): int(v) for k, v in bpe_artifact["token_length"].items()}
vocab_size = int(bpe_artifact["vocab_size"])

assert len(merge_history) == BPE_MERGES
assert vocab_size == 5 + BPE_MERGES

PAD_TOKEN_ID = vocab_size
N_TOKEN_ID = 4
MAX_TOKEN_LENGTH = max(token_length.values())
if MAX_TOKEN_LENGTH >= 256:
    raise ValueError("Compact cache assumes max token length < 256 bp")

TOKEN_LENGTH_LOOKUP = torch.zeros(vocab_size, dtype=torch.int16)
for token_id, length in token_length.items():
    TOKEN_LENGTH_LOOKUP[token_id] = length

MERGE_HASH = hashlib.sha1(repr(merge_history).encode()).hexdigest()[:12]
CACHE_DIR = PROJECT_ROOT / "artifacts" / f"bpe100_pretokenized_{MERGE_HASH}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("BPE merges:", len(merge_history))
print("Vocab:", vocab_size)
print("Max token length:", MAX_TOKEN_LENGTH)
print("Merge hash:", MERGE_HASH)
print("Cache:", CACHE_DIR)


In [ ]:
# Device-agnostic torch BPE. Для pretokenization используем DirectML.

def reverse_complement_storage_codes(raw: torch.Tensor) -> torch.Tensor:
    x = raw.to(torch.int16)
    is_base = x < 8
    case = ((x >= 4) & (x < 8)).to(torch.int16) * 4
    rc = 3 - torch.remainder(x, 4) + case
    rc = torch.where(is_base, rc, torch.full_like(rc, 8))
    return rc.flip(0).to(torch.uint8)


def apply_bpe_once(raw_cpu: torch.Tensor, *, reverse_complement: bool):
    raw = raw_cpu.to(device)
    if reverse_complement:
        raw = reverse_complement_storage_codes(raw)

    x = raw.to(torch.int32)
    tokens = torch.where(x < 8, torch.remainder(x, 4), torch.full_like(x, 4))
    lower = ((x >= 4) & (x < 8)).to(torch.int32)
    del raw, x

    for left, right, new_token in merge_history:
        matches = (tokens[:-1] == left) & (tokens[1:] == right)
        starts = torch.nonzero(matches, as_tuple=False).flatten()
        if starts.numel() == 0:
            continue

        if left == right and starts.numel() > 1:
            new_run = torch.cat([
                torch.tensor([True], dtype=torch.bool, device=starts.device),
                starts[1:] != starts[:-1] + 1,
            ])
            run_ids = torch.cumsum(new_run.to(torch.long), dim=0) - 1
            run_starts = torch.nonzero(new_run, as_tuple=False).flatten()
            offsets = torch.arange(starts.numel(), device=starts.device) - run_starts[run_ids]
            starts = starts[offsets % 2 == 0]

        merged = tokens.clone()
        merged[starts] = new_token
        merged_lower = lower.clone()
        merged_lower[starts] = lower[starts] + lower[starts + 1]

        remove = torch.zeros(tokens.numel(), dtype=torch.bool, device=tokens.device)
        remove[starts + 1] = True
        tokens = merged[~remove]
        lower = merged_lower[~remove]

    token_ids = tokens.to(torch.uint8).cpu().contiguous()
    lower_count = lower.to(torch.uint8).cpu().contiguous()
    lengths = TOKEN_LENGTH_LOOKUP[token_ids.long()].to(torch.int32)

    assert int(lengths.sum()) == raw_cpu.numel()
    assert torch.all(lower_count.to(torch.int16) <= lengths.to(torch.int16))
    return token_ids, lower_count


def safe_name(contig: str):
    stem = re.sub(r"[^A-Za-z0-9._-]+", "_", contig)[:80]
    suffix = hashlib.sha1(contig.encode()).hexdigest()[:10]
    return f"{stem}_{suffix}"


def cache_path(contig: str, orientation: str):
    return CACHE_DIR / f"{safe_name(contig)}.{orientation}.pt"


def pretokenize_contig(contig: str, orientation: str, overwrite=False):
    path = cache_path(contig, orientation)
    if path.is_file() and not overwrite:
        return path

    raw = torch.as_tensor(sequences[contig], dtype=torch.uint8).contiguous()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        with torch.no_grad():
            ids, lower = apply_bpe_once(raw, reverse_complement=(orientation == "rc"))

    fallback = [str(w.message) for w in caught if "fall back" in str(w.message).lower()]
    if fallback:
        raise RuntimeError("DirectML CPU fallback in BPE: " + " | ".join(fallback))

    torch.save({
        "contig": contig,
        "orientation": orientation,
        "length_bp": raw.numel(),
        "token_ids": ids,
        "lowercase_count": lower,
        "merge_hash": MERGE_HASH,
        "bpe_merges": 100,
    }, path)

    print(
        f"{contig} [{orientation}] bp={raw.numel():,} "
        f"tokens={ids.numel():,} compression={raw.numel()/ids.numel():.3f}x"
    )
    del raw, ids, lower
    gc.collect()
    return path


In [ ]:
# Сначала короткий DirectML smoke-test на одном global window.
smoke_contig = train_contigs[0]
smoke_raw = torch.as_tensor(sequences[smoke_contig][:GLOBAL_LENGTH], dtype=torch.uint8).contiguous()
if smoke_raw.numel() < GLOBAL_LENGTH:
    smoke_raw = torch.cat([
        smoke_raw,
        torch.full((GLOBAL_LENGTH - smoke_raw.numel(),), 8, dtype=torch.uint8),
    ])

started = time.perf_counter()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    with torch.no_grad():
        smoke_ids, smoke_lower = apply_bpe_once(smoke_raw, reverse_complement=False)

fallback = [str(w.message) for w in caught if "fall back" in str(w.message).lower()]
print("Input bp:", smoke_raw.numel())
print("Tokens:", smoke_ids.numel())
print("Compression:", round(smoke_raw.numel()/smoke_ids.numel(), 4))
print("Time sec:", round(time.perf_counter()-started, 3))
print("CPU fallback:", fallback)
assert not fallback
print("GPU BPE smoke-test: OK")


In [ ]:
# ONE-TIME cache build. Resume-safe: готовые contigs пропускаются.
# Эту ячейку лучше запускать после завершения старой full validation.
CACHE_PARTS = ("train", "validation")
OVERWRITE_CACHE = False

cache_contigs = []
if "train" in CACHE_PARTS:
    cache_contigs += list(train_contigs)
if "validation" in CACHE_PARTS:
    cache_contigs += list(validation_contigs)
cache_contigs = tuple(dict.fromkeys(cache_contigs))

started = time.perf_counter()
for i, contig in enumerate(cache_contigs, 1):
    print(f"\n[{i}/{len(cache_contigs)}] {contig}")
    pretokenize_contig(contig, "plus", OVERWRITE_CACHE)
    pretokenize_contig(contig, "rc", OVERWRITE_CACHE)

print("\nCache ready. Minutes:", round((time.perf_counter()-started)/60, 2))


In [ ]:
# Load compact cache to RAM and build cumulative token ends once.

def load_cached(contig, orientation):
    payload = torch.load(cache_path(contig, orientation), map_location="cpu", weights_only=False)
    assert payload["merge_hash"] == MERGE_HASH
    ids = payload["token_ids"].to(torch.uint8).contiguous()
    lower = payload["lowercase_count"].to(torch.uint8).contiguous()
    lengths = TOKEN_LENGTH_LOOKUP[ids.long()].to(torch.int16).contiguous()
    ends = torch.cumsum(lengths.to(torch.int32), 0).contiguous()
    assert int(ends[-1]) == int(payload["length_bp"])
    return {
        "length_bp": int(payload["length_bp"]),
        "ids": ids,
        "lower": lower,
        "lengths": lengths,
        "ends": ends,
    }

PRETOKENIZED = {}
started = time.perf_counter()
for contig in cache_contigs:
    PRETOKENIZED[(contig, "plus")] = load_cached(contig, "plus")
    PRETOKENIZED[(contig, "rc")] = load_cached(contig, "rc")

train_bp = sum(len(sequences[c]) for c in train_contigs)
train_tokens = sum(PRETOKENIZED[(c,"plus")]["ids"].numel() for c in train_contigs)
print("Orientations in RAM:", len(PRETOKENIZED))
print("Load sec:", round(time.perf_counter()-started, 2))
print("Train compression:", round(train_bp/train_tokens, 4), "x")


In [ ]:
# Fast slicing: no BPE merges here.

def slice_cached(contig, genomic_start, genomic_end, *, rc=False):
    item = PRETOKENIZED[(contig, "rc" if rc else "plus")]
    L = item["length_bp"]

    if rc:
        start, end = L - genomic_end, L - genomic_start
    else:
        start, end = genomic_start, genomic_end
    assert end - start == GLOBAL_LENGTH

    left_pad = max(0, -start)
    right_pad = max(0, end - L)
    a, b = max(0, start), min(L, end)
    ends = item["ends"]

    if b > a:
        first = int(torch.searchsorted(ends, torch.tensor(a, dtype=ends.dtype), right=True))
        last = int(torch.searchsorted(ends, torch.tensor(b-1, dtype=ends.dtype), right=True))
        stop = last + 1
        core_ids = item["ids"][first:stop]
        core_lower = item["lower"][first:stop]
        core_lengths = item["lengths"][first:stop]
        coords = torch.arange(a, b, dtype=ends.dtype)
        core_map = (torch.searchsorted(ends, coords, right=True) - first + left_pad).long()
    else:
        core_ids = torch.empty(0, dtype=torch.uint8)
        core_lower = torch.empty(0, dtype=torch.uint8)
        core_lengths = torch.empty(0, dtype=torch.int16)
        core_map = torch.empty(0, dtype=torch.long)

    left_ids = torch.full((left_pad,), N_TOKEN_ID, dtype=torch.uint8)
    left_lower = torch.zeros(left_pad, dtype=torch.uint8)
    left_lengths = torch.ones(left_pad, dtype=torch.int16)
    left_map = torch.arange(left_pad, dtype=torch.long)

    right_ids = torch.full((right_pad,), N_TOKEN_ID, dtype=torch.uint8)
    right_lower = torch.zeros(right_pad, dtype=torch.uint8)
    right_lengths = torch.ones(right_pad, dtype=torch.int16)
    right_map = left_pad + core_ids.numel() + torch.arange(right_pad, dtype=torch.long)

    ids = torch.cat([left_ids, core_ids, right_ids])
    lower = torch.cat([left_lower, core_lower, right_lower])
    lengths = torch.cat([left_lengths, core_lengths, right_lengths])
    bp_to_token = torch.cat([left_map, core_map, right_map])
    assert bp_to_token.numel() == GLOBAL_LENGTH

    return {
        "ids": ids,
        "lengths": lengths,
        "softmask": lower.float()/lengths.float(),
        "bp_to_token": bp_to_token,
    }


def build_cached_both_batch(tiles):
    items = []
    for rc in (False, True):
        for i in range(len(tiles)):
            tile = tiles.iloc[i]
            item = slice_cached(
                str(tile["contig"]),
                int(tile["target_start"]) - GLOBAL_FLANK,
                int(tile["target_end"]) + GLOBAL_FLANK,
                rc=rc,
            )
            item["local_map"] = item["bp_to_token"][LOCAL_START_IN_GLOBAL:LOCAL_END_IN_GLOBAL]
            items.append(item)

    T = max(x["ids"].numel() for x in items)
    N = len(items)
    token_ids = torch.full((N,T), PAD_TOKEN_ID, dtype=torch.long)
    token_lengths = torch.zeros((N,T), dtype=torch.float32)
    softmask = torch.zeros((N,T), dtype=torch.float32)
    token_mask = torch.zeros((N,T), dtype=torch.float32)
    local_map = torch.empty((N,LOCAL_LENGTH), dtype=torch.long)

    for row, x in enumerate(items):
        n = x["ids"].numel()
        token_ids[row,:n] = x["ids"].long()
        token_lengths[row,:n] = x["lengths"].float()
        softmask[row,:n] = x["softmask"]
        token_mask[row,:n] = 1.0
        local_map[row] = x["local_map"]

    return {
        "token_ids": token_ids.contiguous(),
        "token_lengths": token_lengths.contiguous(),
        "softmask": softmask.contiguous(),
        "token_mask": token_mask.contiguous(),
        "local_map": local_map.contiguous(),
    }

smoke_batch = train_generator.sample_batch(batch_size=4)
t0 = time.perf_counter()
smoke_global = build_cached_both_batch(smoke_batch.tiles)
print("Cached batch build ms:", round((time.perf_counter()-t0)*1000, 2))
print("Token batch:", smoke_global["token_ids"].shape)
assert smoke_global["token_ids"].shape[0] == 8


In [ ]:
GLOBAL_DILATIONS = (1,2,4,8,16,32,64,128,256,512)

class GlobalResidualBlock(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=dilation, dilation=dilation)
        self.act = nn.GELU()
    def forward(self, x, mask):
        m = mask.unsqueeze(1).to(dtype=x.dtype)
        residual = x
        x = self.act(self.conv1(x)) * m
        return self.act(self.conv2(x) + residual) * m

class GlobalBPEEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size+1, 16, padding_idx=PAD_TOKEN_ID)
        self.proj = nn.Sequential(nn.Conv1d(18,32,1), nn.GELU())
        self.blocks = nn.ModuleList([GlobalResidualBlock(32,d) for d in GLOBAL_DILATIONS])
    def forward(self, ids, lengths, softmask, mask):
        emb = self.embedding(ids)
        length_feature = (lengths.to(emb.dtype)/MAX_TOKEN_LENGTH).unsqueeze(-1)
        soft_feature = softmask.to(emb.dtype).unsqueeze(-1)
        x = torch.cat([emb, soft_feature, length_feature], -1)
        x = x * mask.unsqueeze(-1).to(x.dtype)
        x = self.proj(x.transpose(1,2).contiguous())
        x = x * mask.unsqueeze(1).to(x.dtype)
        for block in self.blocks:
            x = block(x, mask)
        return x

class WideBPE100Model(nn.Module):
    def __init__(self):
        super().__init__()
        local = create_cnn_presence_intensity()
        self.backbone = local.backbone
        self.global_encoder = GlobalBPEEncoder()
        self.fusion = nn.Sequential(nn.Conv1d(64,32,1), nn.GELU())
        self.profile_head = nn.Conv1d(32,2,1)
    def forward(self, x_local, ids, lengths, softmask, mask, local_map):
        local = self.backbone(x_local)
        global_features = self.global_encoder(ids, lengths, softmask, mask)
        gather_index = local_map.unsqueeze(1).expand(-1,32,-1).contiguous()
        global_local = torch.gather(global_features, 2, gather_index)
        out = self.profile_head(self.fusion(torch.cat([local,global_local],1)))
        return out[:,0:1,:], out[:,1:2,:]


def create_model():
    return WideBPE100Model()


def forward_both(model, batch, global_batch, *, grad=True):
    B = len(batch.tiles)
    x = batch.x_both_strands.contiguous().to(device)
    g = {k: v.contiguous().to(device) for k,v in global_batch.items()}
    ctx = torch.enable_grad() if grad else torch.no_grad()
    with ctx:
        p, q = model(x, g["token_ids"], g["token_lengths"], g["softmask"], g["token_mask"], g["local_map"])
        p = p[:,:,LOCAL_FLANK:-LOCAL_FLANK]
        q = q[:,:,LOCAL_FLANK:-LOCAL_FLANK]
    return p[:B], p[B:], q[:B], q[B:]

model = create_model().to(device)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
model.eval()
outputs = forward_both(model, smoke_batch, smoke_global, grad=False)
for name,t in zip(("plus p","minus p reverse","plus intensity","minus intensity reverse"), outputs):
    print(name, t.shape)
    assert t.shape == (4,1,1024)
print("Combined PLUS+RC forward: OK")


In [ ]:
# One real backward step before the long run.
INTENSITY_LOSS_WEIGHT = 0.1
model.train()
optimizer = DirectMLAdam(model.parameters(), lr=1e-3)
optimizer.zero_grad(set_to_none=True)

batch = smoke_batch
plus_count = batch.count[:,0:1,:].contiguous()
minus_count_reverse = batch.count[:,1:2,:].flip(-1).contiguous()

plus_target = batch.plus_target.contiguous().to(device)
minus_target = batch.minus_target_reverse.contiguous().to(device)
plus_w = batch.plus_loss_weight.contiguous().to(device)
minus_w = batch.minus_loss_weight_reverse.contiguous().to(device)
plus_i_target = torch.log1p(plus_count).to(device)
minus_i_target = torch.log1p(minus_count_reverse).to(device)
plus_i_w = (2*batch.plus_loss_weight*(plus_count>0).float()).to(device)
minus_i_w = (2*batch.minus_loss_weight_reverse*(minus_count_reverse>0).float()).to(device)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    pp, mp, pi, mi = forward_both(model, batch, smoke_global, grad=True)
    presence_loss = dml_weighted_profile_bce_with_logits(pp,mp,plus_target,minus_target,plus_w,minus_w)
    intensity_loss = dml_weighted_huber(pi,plus_i_target,plus_i_w) + dml_weighted_huber(mi,minus_i_target,minus_i_w)
    total_loss = presence_loss + INTENSITY_LOSS_WEIGHT*intensity_loss
    total_loss.backward()
    optimizer.step()

fallback = [str(w.message) for w in caught if "fall back" in str(w.message).lower()]
print("Presence:", presence_loss.item())
print("Intensity:", intensity_loss.item())
print("Total:", total_loss.item())
print("Global embedding grad:", model.global_encoder.embedding.weight.grad.abs().sum().item())
print("CPU fallback:", fallback)
assert not fallback


In [ ]:
# 1000-step BPE sweep run.
STEPS = 1_000
BATCH_SIZE = 4
LOG_EVERY = 100
SEED = 42
INTENSITY_LOSS_WEIGHT = 0.1

RUN_DIR = PROJECT_ROOT / "artifacts" / "experiments" / "CNN_EXP007" / "run_001"
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = RUN_DIR / "checkpoint_step_01000.pt"
HISTORY_PATH = RUN_DIR / "training_history.csv"

torch.manual_seed(SEED)
generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=SEED,
)
model = create_model().to(device)
optimizer = DirectMLAdam(model.parameters(), lr=1e-3)
model.train()

history=[]
ema_p=ema_i=ema_t=None
seen=0
prep_seconds=0.0
started=time.perf_counter()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    for step in range(1,STEPS+1):
        t0=time.perf_counter()
        batch=generator.sample_batch(batch_size=BATCH_SIZE)
        gb=build_cached_both_batch(batch.tiles)
        prep_seconds += time.perf_counter()-t0

        plus_count=batch.count[:,0:1,:].contiguous()
        minus_count=batch.count[:,1:2,:].flip(-1).contiguous()
        plus_target=batch.plus_target.to(device)
        minus_target=batch.minus_target_reverse.to(device)
        plus_w=batch.plus_loss_weight.to(device)
        minus_w=batch.minus_loss_weight_reverse.to(device)
        plus_i_target=torch.log1p(plus_count).to(device)
        minus_i_target=torch.log1p(minus_count).to(device)
        plus_i_w=(2*batch.plus_loss_weight*(plus_count>0).float()).to(device)
        minus_i_w=(2*batch.minus_loss_weight_reverse*(minus_count>0).float()).to(device)

        optimizer.zero_grad(set_to_none=True)
        pp,mp,pi,mi=forward_both(model,batch,gb,grad=True)
        lp=dml_weighted_profile_bce_with_logits(pp,mp,plus_target,minus_target,plus_w,minus_w)
        li=dml_weighted_huber(pi,plus_i_target,plus_i_w)+dml_weighted_huber(mi,minus_i_target,minus_i_w)
        loss=lp+INTENSITY_LOSS_WEIGHT*li
        value=loss.item()
        if not math.isfinite(value):
            raise FloatingPointError(f"Non-finite loss at step {step}")
        loss.backward()
        optimizer.step()

        pv,iv=lp.item(),li.item()
        if ema_t is None:
            ema_p,ema_i,ema_t=pv,iv,value
        else:
            ema_p=.95*ema_p+.05*pv
            ema_i=.95*ema_i+.05*iv
            ema_t=.95*ema_t+.05*value
        seen += int(batch.mask.sum())
        elapsed=time.perf_counter()-started
        history.append({
            "step":step,"presence_loss":pv,"intensity_loss":iv,"total_loss":value,
            "ema_presence":ema_p,"ema_intensity":ema_i,"ema_total":ema_t,
            "positions_per_second":seen/elapsed,"prep_fraction":prep_seconds/elapsed,
        })
        if step==1 or step%LOG_EVERY==0:
            print(
                f"step={step:4d} presence={pv:.4f} intensity={iv:.4f} total={value:.4f} "
                f"ema=({ema_p:.4f},{ema_i:.4f},{ema_t:.4f}) "
                f"positions/s={seen/elapsed:,.0f} prep={100*prep_seconds/elapsed:.1f}%"
            )

fallback=[str(w.message) for w in caught if "fall back" in str(w.message).lower()]
if fallback:
    raise RuntimeError("DirectML fallback: "+" | ".join(fallback))

pd.DataFrame(history).to_csv(HISTORY_PATH,index=False)
torch.save({
    "step":STEPS,
    "model_state_dict":{k:v.detach().cpu() for k,v in model.state_dict().items()},
    "bpe_merges":100,"merge_hash":MERGE_HASH,
    "global_flank":GLOBAL_FLANK,"global_length":GLOBAL_LENGTH,
    "final_ema":{"presence":ema_p,"intensity":ema_i,"total":ema_t},
},CHECKPOINT_PATH)

print("Elapsed min:",round((time.perf_counter()-started)/60,2))
print("Final EMA:",ema_p,ema_i,ema_t)
print("Checkpoint:",CHECKPOINT_PATH)


In [ ]:
# Fixed validation subset for BPE-50/100/200/400 sweep.
SWEEP_VALIDATION_TILES = 5_000
all_tiles = validation_generator.tile_index
n = min(SWEEP_VALIDATION_TILES, len(all_tiles))
if n == len(all_tiles):
    sweep_tiles = all_tiles.copy()
else:
    idx = torch.linspace(0,len(all_tiles)-1,steps=n).round().long().unique().tolist()
    sweep_tiles = all_tiles.iloc[idx].copy()

print("Tiles:",len(sweep_tiles))
print("Position-strand:",int(sweep_tiles["allowed_position_strand"].sum()))
print("Positives:",int(sweep_tiles["positive_positions"].sum()))
print("Contigs:",sweep_tiles["contig"].nunique())


In [ ]:
# Validation — same score construction and metric accumulator as the old evaluator.
import numpy as np
from scipy.special import expit
import ml_project.epic_cnn.evaluation as epic_eval


def evaluate_cached(model, tiles, batch_size=8, progress_every=100):
    score_names=("presence_probability","intensity_prediction","expected_signal")
    contigs=tuple(str(x) for x in tiles["contig"].unique())
    scale=100_000
    levels=100_001
    acc={name:{c:epic_eval._MetricAccumulator(levels) for c in contigs} for name in score_names}
    positions=positives=0
    started=time.perf_counter()
    model.eval()

    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        for batch_no,start in enumerate(range(0,len(tiles),batch_size),1):
            selected=tiles.iloc[start:start+batch_size]
            batch=validation_generator.make_batch(selected)
            gb=build_cached_both_batch(batch.tiles)
            pp,mp,pi,mi=forward_both(model,batch,gb,grad=False)

            p_logits=np.stack((pp.cpu().numpy()[:,0,:],mp.cpu().numpy()[:,0,::-1]),axis=1)
            i_raw=np.stack((pi.cpu().numpy()[:,0,:],mi.cpu().numpy()[:,0,::-1]),axis=1)
            p_prob=expit(p_logits)
            i_score=expit(i_raw)
            expected_raw=p_prob*np.expm1(np.clip(i_raw,0,20))
            scores={
                "presence_probability":p_prob,
                "intensity_prediction":i_score,
                "expected_signal":expected_raw/(1+expected_raw),
            }
            mask=batch.mask.numpy(); counts=batch.count.numpy()
            cs=selected["contig"].astype(str).to_numpy()
            for name,score in scores.items():
                q=np.rint(np.clip(score,0,1)*scale).astype(np.int32)
                for c in np.unique(cs):
                    rows=np.flatnonzero(cs==c)
                    acc[name][c].add(score[rows],q[rows],mask[rows],counts[rows])
            positions += int(mask.sum())
            positives += int((mask & (counts>0)).sum())
            if batch_no==1 or batch_no%progress_every==0 or start+len(selected)>=len(tiles):
                elapsed=time.perf_counter()-started
                print(f"tiles={start+len(selected):>5}/{len(tiles)} positions={positions:>9} positions/s={positions/elapsed:,.0f}")

    fallback=tuple(str(w.message) for w in caught if "fall back" in str(w.message).lower())
    if fallback:
        raise RuntimeError("DirectML fallback: "+" | ".join(fallback))
    assert positions==int(tiles["allowed_position_strand"].sum())
    assert positives==int(tiles["positive_positions"].sum())

    rows=[]
    for name in score_names:
        overall=epic_eval._MetricAccumulator(levels)
        overall.merge(list(acc[name].values()))
        row=overall.metrics("overall")
        row["score"]=name
        rows.append(row)
    metrics=pd.DataFrame(rows)[[
        "score","segment","position_strand","positives","prevalence",
        "average_precision","ap_lift","epic_spearman","quantized_spearman"
    ]]
    elapsed=time.perf_counter()-started
    return metrics,elapsed,positions/elapsed

checkpoint=torch.load(CHECKPOINT_PATH,map_location="cpu",weights_only=False)
validation_model=create_model().to(device)
validation_model.load_state_dict(checkpoint["model_state_dict"])
metrics,seconds,speed=evaluate_cached(validation_model,sweep_tiles,batch_size=8)
METRICS_PATH=RUN_DIR/"validation_sweep_5000_tiles.csv"
metrics.to_csv(METRICS_PATH,index=False)
display(metrics)
print("Validation min:",round(seconds/60,2))
print("Positions/s:",f"{speed:,.0f}")
print("Metrics:",METRICS_PATH)


In [ ]:
# Full validation only for a finalist.
RUN_FULL_VALIDATION = False
if RUN_FULL_VALIDATION:
    full_metrics,full_seconds,full_speed=evaluate_cached(
        validation_model,
        validation_generator.tile_index,
        batch_size=8,
        progress_every=500,
    )
    path=RUN_DIR/"validation_metrics_full.csv"
    full_metrics.to_csv(path,index=False)
    display(full_metrics)
    print("Full validation min:",round(full_seconds/60,2))
    print("Positions/s:",f"{full_speed:,.0f}")
else:
    print("Full validation disabled for sweep.")
